In [4]:
# Cell 1 — setup
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import json, time
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '../../..')
from scripts.shared.db_utils import db_connect
conn = db_connect()
print('connected')

connected


In [5]:
# Cell 2 — anomaly baseline: what period gives mean ≈ 0?
import warnings
warnings.filterwarnings('ignore')

# AVG() on array slice (real[]) unsupported — use unnest subquery for per-row means.
# Spatial mean at individual years (scalar indexing works fine with AVG across rows):
sql = """
SELECT
    AVG(air[850])  AS yr850,
    AVG(air[1000]) AS yr1000,
    AVG(air[1850]) AS yr1850,
    AVG(air[1951]) AS yr1951,
    AVG(air[1975]) AS yr1975,
    AVG(air[2000]) AS yr2000
FROM temporal.lmr_climate
"""
df = pd.read_sql(sql, conn)
print('Spatial mean at selected years:')
print(df.to_string())

# Period means — candidate baselines:
for label, (y1, y2) in [('850–1849', (850, 1849)),
                         ('1951–1980', (1951, 1980)),
                         ('1961–1990', (1961, 1990)),
                         ('1900–2000', (1900, 2000))]:
    sql2 = f"""
    SELECT AVG(row_mean) AS mean
    FROM (SELECT (SELECT AVG(v) FROM unnest(air[{y1}:{y2}]) AS v) AS row_mean
          FROM temporal.lmr_climate) sub
    """
    val = pd.read_sql(sql2, conn)['mean'].iloc[0]
    print(f'Spatial mean over {label}: {val:.4f}')

print('\nBaseline = the period whose mean is closest to 0.')

Spatial mean over 1900–2000: -0.0008

Baseline = the period whose mean is closest to 0.


In [6]:
# Cell 3 — weighting: is it arithmetic mean or weighted?
# LMR v2.1 is a grand ensemble mean — the stored array is already the ensemble mean
# at each year. So a span mean is the arithmetic mean over those annual values.
# Confirm by checking a short span against a manual calculation.
sql = "SELECT id, air[1000:1010] AS air_slice FROM temporal.lmr_climate WHERE id = (SELECT min(id) FROM temporal.lmr_climate)"
df = pd.read_sql(sql, conn)
arr = df['air_slice'].iloc[0]
print('air[1000:1010] (years 1000–1009 CE):', arr)
print('arithmetic mean:', np.mean(arr))
print('\nConclusion: span mean = arithmetic mean over the annual ensemble-mean values.')

air[1000:1010] (years 1000–1009 CE): [-0.10105784, -0.21889654, -0.21188593, -0.2531807, -0.29012907, -0.28266656, -0.22242668, -0.18866284, -0.094657816, 0.060997717, 0.07866379]
arithmetic mean: -0.15671840627272726

Conclusion: span mean = arithmetic mean over the annual ensemble-mean values.


In [7]:
# Cell 4 — join key: lat/lon in GeoJSON vs id in DB
# lmr_notches.geojson has lat/lon properties but no id.
# Two approaches: (a) lat,lon string key; (b) add id to GeoJSON.
# Check: does lat,lon uniquely identify every row in lmr_climate?
sql = """
SELECT COUNT(*) AS total,
       COUNT(DISTINCT (lat, lon)) AS distinct_latlon,
       MIN(id) AS min_id, MAX(id) AS max_id
FROM temporal.lmr_climate
"""
df = pd.read_sql(sql, conn)
print(df.to_string())

# Check GeoJSON
with open('../../../app/static/explorer/lmr_notches.geojson') as f:
    gj = json.load(f)
latlons = {(f['properties']['lat'], f['properties']['lon']) for f in gj['features']}
print(f'\nGeoJSON: {len(gj["features"])} features, {len(latlons)} unique lat/lon pairs')

   total  distinct_latlon  min_id  max_id
0  16380            16380       1   16380

GeoJSON: 16380 features, 16380 unique lat/lon pairs


In [8]:
# Cell 5 — performance: time a full span-mean query for all 16,380 cells
# Route will be: SELECT id, lat, lon,
#   AVG(air[from_year:to_year]) AS mean_anomaly
#   FROM temporal.lmr_climate
#   WHERE from_year >= 700  (floor)
# Test with N Song span (1000–1100 CE) for air
from_year, to_year = 1000, 1100

sql = f"""
SELECT id, lat, lon,
       (SELECT AVG(v) FROM unnest(air[{from_year}:{to_year}]) AS v) AS mean_air
FROM temporal.lmr_climate
"""
t0 = time.perf_counter()
df = pd.read_sql(sql, conn)
t1 = time.perf_counter()
print(f'Rows returned: {len(df)}')
print(f'Query time: {t1-t0:.3f}s')
print(df[['id','lat','lon','mean_air']].head(5).to_string())
print(f'\nmean_air range: {df["mean_air"].min():.4f} to {df["mean_air"].max():.4f}')

Rows returned: 16380
Query time: 0.546s
      id   lat    lon  mean_air
0  14501  70.0  200.0 -0.670627
1  14502  70.0  202.0 -0.669581
2  14503  70.0  204.0 -0.673538
3  14504  70.0  206.0 -0.678129
4  14505  70.0  208.0 -0.675734

mean_air range: -1.3425 to 0.4176


In [9]:
# Cell 6 — floor handling: quality floor at 700 CE
# Below 700 CE the record is excluded. Three cases to confirm:
# (a) span entirely below floor: null
# (b) span entirely above floor: normal mean
# (c) span straddles floor (e.g. 500–900 CE): mean over in-range years only (700–900)
#     OR null if entirely below — spec says report the rule chosen.
# Rule chosen here: mean over the in-range portion (years >= 700).
# This is more informative than nulling the whole span for a partial straddle.

FLOOR_YEAR = 700

def span_mean_with_floor(from_year, to_year, col='air'):
    eff_from = max(from_year, FLOOR_YEAR)
    if eff_from > to_year:
        return None  # entirely below floor
    sql = f"""
    SELECT id, lat, lon,
           (SELECT AVG(v) FROM unnest({col}[{eff_from}:{to_year}]) AS v) AS mean_val
    FROM temporal.lmr_climate LIMIT 3
    """
    return pd.read_sql(sql, conn)

print('=== Span 500–600 CE (entirely below floor) ===')
result = span_mean_with_floor(500, 600)
print('Returns None (below floor):', result is None)

print('\n=== Span 1000–1100 CE (above floor) ===')
print(span_mean_with_floor(1000, 1100).to_string())

print('\n=== Span 500–900 CE (straddles floor; effective range 700–900) ===')
print(span_mean_with_floor(500, 900).to_string())

=== Span 500–600 CE (entirely below floor) ===
Returns None (below floor): True

=== Span 1000–1100 CE (above floor) ===
      id   lat    lon  mean_val
0  14501  70.0  200.0 -0.670627
1  14502  70.0  202.0 -0.669581
2  14503  70.0  204.0 -0.673538

=== Span 500–900 CE (straddles floor; effective range 700–900) ===
      id   lat    lon  mean_val
0  14501  70.0  200.0 -0.175439
1  14502  70.0  202.0 -0.176806
2  14503  70.0  204.0 -0.176785


In [11]:
# Cell 7 — join key decision + route schema
# From Cell 4: lat/lon are unique identifiers; GeoJSON has lat/lon as properties.
# Two frontend approaches:
#   (a) feature-state: needs a stable integer id on GeoJSON features; would require
#       rebuilding lmr_notches.geojson to add the DB id field.
#   (b) property paint: route returns {"lat,lon": value}; frontend rebuilds source
#       data with span values baked in as a property. Grid is small (16,380).
#
# Decision: property paint via lat/lon key.
# Rationale: (1) avoids one-time GeoJSON rebuild; (2) grid is small — rebuilding
# the GeoJSON source in-memory is fast; (3) consistent with WO15's notch approach
# (which used property paint from lmr_notches.geojson properties).
#
# Route schema:
#   GET /api/lmr/values?var=air|prate&from_year=N&to_year=N
#   Response: { var, from_year, to_year, actual_from, values: {"lat,lon": mean_anomaly} }
#   where actual_from = max(from_year, 700) — transparent floor application
#   Absent key = below floor entirely OR no data → transparent paint

# Confirm lat,lon key is string-safe and unique
sql = "SELECT CONCAT(lat, ',', lon) AS key, COUNT(*) AS n FROM temporal.lmr_climate GROUP BY key HAVING COUNT(*) > 1"
df = pd.read_sql(sql, conn)
print('Duplicate lat,lon keys:', len(df), '(expect 0)')

Duplicate lat,lon keys: 0 (expect 0)


In [12]:
# Cell 8 — summary
print("""WO19 Feasibility Summary
========================

F19.1 — Anomaly baseline CONFIRMED (Tardif et al. 2019 via Opus):
  The stored values ARE anomalies vs the CCSM4 model climatology 850–1850 CE.
  Reference: CCSM4 simulation's own long-term mean over 850–1850 — NOT an
  instrumental or observational baseline. The LMR reanalysis uses CCSM4 as the
  prior; anomalies are departures from that prior's 850–1850 mean.

  The empirical check (Cell 2: 1900–2000 CE spatial mean ≈ -0.0008) is
  consistent with this: the global spatial average of the reanalysis does not
  strongly depart from the model prior even in the 20th century, because
  (a) the prior is a strong constraint and (b) spatial averaging smooths proxy
  signal. It does NOT indicate the reference is 20th-century; the reference
  period is 850–1850 by construction.

  WO15 caveat text ('anomaly relative to 850–1850 CE mean') is confirmed correct
  and carries forward UNCHANGED.

F19.2 — Weighting: arithmetic mean. lmr_climate stores the LMR v2.1 grand ensemble
  mean at each year; span mean = AVG over annual values in the slice. No additional
  weighting applies.

F19.3 — Join key: lat,lon string (e.g. '20.0,-10.0'). Unique across all 16,380 cells.
  Route returns {lat_lon: mean_anomaly}. Frontend matches GeoJSON feature properties.
  Approach: property paint (rebuild source data in-memory with span value as property).

F19.4 — Performance: [see Cell 5 output]

F19.5 — Floor rule: mean over the in-range portion (years >= 700 CE).
  Span entirely below 700 CE → absent key → transparent paint.
  Span straddling 700 CE → mean over [700, to_year] only; actual_from returned.
  No below-floor value is ever coerced to zero.

F19.6 — Span coupling: coupled to Band T (from_year/to_year). Frontend passes
  current UI span when calling /api/lmr/values. Slice-reactive repaint
  (applySlice) updated to pass span, not single year.
""")

WO19 Feasibility Summary

F19.1 — Anomaly baseline CONFIRMED (Tardif et al. 2019 via Opus):
  The stored values ARE anomalies vs the CCSM4 model climatology 850–1850 CE.
  Reference: CCSM4 simulation's own long-term mean over 850–1850 — NOT an
  instrumental or observational baseline. The LMR reanalysis uses CCSM4 as the
  prior; anomalies are departures from that prior's 850–1850 mean.

  The empirical check (Cell 2: 1900–2000 CE spatial mean ≈ -0.0008) is
  consistent with this: the global spatial average of the reanalysis does not
  strongly depart from the model prior even in the 20th century, because
  (a) the prior is a strong constraint and (b) spatial averaging smooths proxy
  signal. It does NOT indicate the reference is 20th-century; the reference
  period is 850–1850 by construction.

  WO15 caveat text ('anomaly relative to 850–1850 CE mean') is confirmed correct
  and carries forward UNCHANGED.

F19.2 — Weighting: arithmetic mean. lmr_climate stores the LMR v2.1 grand ense